# LunarLander-v3 — a reinforcement-learning autopilot, and what its number means

This notebook is the work in order: the three guided exercises, then the mission. It is
committed **without outputs** (`nbstripout`), so nothing here asserts a result in prose that
you would have to take on trust. Every figure is read from an artefact under `data/` when
the cell runs, and every one of those artefacts is regenerated by a script in `scripts/`.

Two things this notebook is careful about, because the first version was not:

- **the published number is about a method, not about one run.** Five independent PPO
  trainings, and the spread between them published beside the mean;
- **the hyper-parameter study exists.** It used to be a table of `~280`, `< 200`, `~270`
  with nothing behind it. It is now a CSV, and it says out loud what one seed per trial can
  and cannot support.

The logic lives in `src/rl_lander/`. This notebook orchestrates and reports.


## 0. Setup


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from rl_lander.utils import DATA_DIR, DEFAULT_MODEL_PATH, EVALUATION_CSV, ensure_dirs, set_global_seed

ensure_dirs()
set_global_seed(42)

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"gymnasium {gym.__version__}")


def read_artifact(name: str):
    """Read a published artefact, or say plainly which command produces it."""
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing. Run `uv run python scripts/train_study.py`, then "
            "`uv run python scripts/aggregate_study.py`."
        )
    if path.suffix == ".json":
        return json.loads(path.read_text(encoding="utf-8"))
    return pd.read_csv(path)


## 1. Exercise 1 — the loop, with a random policy

The `observation → action → reward` cycle on `CartPole-v1`: a continuous observation space
(`Box(4,)`) and a discrete action space (`Discrete(2)`). The policy draws uniformly from the
action space, which is the baseline every learned policy has to beat.


In [ ]:
from rl_lander.exercises.exercise1_cartpole import describe_spaces, run_random_policy

env = gym.make("CartPole-v1")
for key, value in describe_spaces(env).items():
    print(f"{key:<22}: {value}")
env.close()


In [ ]:
history = run_random_policy(env_id="CartPole-v1", n_episodes=10, seed=42)
rewards_random = np.array([h.total_reward for h in history])

print(f"mean reward over {len(history)} episodes: {rewards_random.mean():.1f} "
      f"(min {rewards_random.min():.0f}, max {rewards_random.max():.0f})")
print(f"CartPole-v1 counts as solved at a mean of 475 over 100 consecutive episodes.")


**One thing worth noticing in the code rather than in the number.** `run_random_policy`
seeds the action space as well as the environment. It did not, and the consequence was
quiet: `env.reset(seed=n)` fixes the initial states, while `action_space.sample()` draws
from a *different* generator. A run advertised as seeded was reproducible in its starting
conditions and random in its policy, so two runs of the same "experiment" returned different
rewards and nothing in the output said why.


## 2. Exercise 2 — tabular Q-learning on FrozenLake-v1

The deterministic 4×4 grid has 16 states, so the table $Q \in \mathbb{R}^{16 \times 4}$ fits
without approximation. The update is Bellman's:

$$Q(s, a) \leftarrow Q(s, a) + \alpha\,\bigl(r + \gamma \max_{a'} Q(s', a') - Q(s, a)\bigr).$$

With one correction the first version did not make: **there is no bootstrap through a
terminal state.** When the episode ends there is no next action to take, so the target is
the reward alone. On this grid the error was numerically invisible — the goal state's row
stays at zero because it is never a starting state — which is precisely why it is worth
fixing rather than leaving: it is wrong in general and right by accident here.


In [ ]:
from rl_lander.exercises.exercise2_qlearning import QLearningConfig, evaluate, train

cfg = QLearningConfig(n_episodes=20_000, seed=42)
artefacts = train(cfg)
metrics = evaluate(artefacts["q_table"], n_episodes=100)

print(f"success rate over {metrics['total_episodes']} episodes: {metrics['success_rate']:.0%}")
print(f"mean episode length: {metrics['average_steps']:.1f}")


In [ ]:
rewards = np.array(artefacts["rewards"])
window = 500
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].plot(pd.Series(rewards).rolling(window).mean(), color="#1f77b4")
axes[0].set_title(f"Rolling mean over {window} episodes")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Reward")
axes[0].grid(alpha=0.3)

axes[1].plot(artefacts["epsilons"], color="#d62728")
axes[1].set_title("Exploration schedule")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("epsilon")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
print("Learned Q-table (rows = states, columns = left, down, right, up):\n")
print(np.round(artefacts["q_table"], 3))


The exploration schedule on the right is the second correction. It used to be indexed on the
episode *about to start* rather than on the number finished, so the first two episodes shared
`epsilon = epsilon_start` and the schedule ran one episode behind its own definition for the
whole of training. On 20 000 episodes it changes nothing measurable; it is still a schedule
that did not do what it said.

The tabular approach works here because the state space is discrete and small. That is the
whole point of the exercise, and the reason the next one exists.


## 3. Exercise 3 — Deep Q-Network on CartPole-v1

When the state space is continuous, the table is replaced by a network. Two implementations,
for comparison: a manual PyTorch loop (MLP, replay buffer, target network) and the
Stable-Baselines3 equivalent.


In [ ]:
from rl_lander.exercises.exercise3_dqn import DQNConfig, evaluate_manual_dqn, train_manual_dqn

manual_cfg = DQNConfig(n_episodes=300, seed=42)
policy_net, dqn_result = train_manual_dqn("CartPole-v1", manual_cfg)
manual_metrics = evaluate_manual_dqn(policy_net, n_episodes=20)

print(f"manual DQN — evaluation reward {manual_metrics['mean_reward']:.1f} "
      f"+/- {manual_metrics['std_reward']:.1f} over {manual_metrics['n_episodes']:.0f} episodes")
print(f"final epsilon: {dqn_result.final_epsilon:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(dqn_result.rewards, alpha=0.35, label="Reward per episode")
ax.plot(pd.Series(dqn_result.rewards).rolling(20).mean(), color="#d62728", label="Rolling mean (20)")
ax.axhline(475, color="#2ca02c", linestyle="--", label="CartPole solved threshold (475)")
ax.set_title("Manual DQN on CartPole-v1")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from rl_lander.exercises.exercise3_dqn import SB3DQNConfig, train_sb3_dqn

sb3_cfg = SB3DQNConfig(total_timesteps=25_000, seed=42)
_, sb3_metrics = train_sb3_dqn("CartPole-v1", sb3_cfg)

print(f"SB3 DQN — evaluation reward {sb3_metrics['mean_reward']:.1f} "
      f"+/- {sb3_metrics['std_reward']:.1f} over 100 episodes")


**What this comparison is for, and what it is not.** Both agents learn — the curve climbs
away from the random baseline of section 1. Neither is trained to the point where CartPole-v1
counts as *solved*, which is a mean of 475 over 100 consecutive episodes; the budgets here
are 300 episodes and 25 000 steps, chosen so the notebook runs in minutes. The first version
of this section concluded "both DQNs learn the task", which is true, and then let the reader
hear "solved", which is not. Read the number the cells print against the 475 line on the
plot.

The comparison that matters for the rest of the notebook is a different one: the manual loop
is about a hundred lines that have to be right — the replay buffer, the target network, the
terminal masking, when the first gradient step happens — and the library version is four. Two
of those details were wrong in the manual loop and are pinned by tests now
(`tests/test_dqn_exercise.py`): `learning_starts` was declared and never read, so training
began at `batch_size` transitions; and the target network was synchronised every 20
*episodes*, which on CartPole is anywhere from 200 to 10 000 gradient steps and *shrinks* as
the agent improves — the opposite of what a target network is for.


## 4. The mission — LunarLander-v3


In [ ]:
from rl_lander.training.environments import LUNAR_LANDER_ID, make_eval_env

env = make_eval_env()
obs, _info = env.reset(seed=0)

print(f"environment      : {LUNAR_LANDER_ID}")
print(f"observation space: {env.observation_space}")
print(f"action space     : {env.action_space}")
print(f"initial observation: {np.round(obs, 3)}")
env.close()


The observation is 8-dimensional: `(x, y, vx, vy, angle, angular velocity, left leg contact,
right leg contact)`, the last two boolean. The action is discrete: do nothing, fire left, fire
main, fire right.

The reward, from the Gymnasium documentation:

* between +100 and +140 for moving from the top of the screen to the landing pad and coming
  to rest;
* −0.3 per frame with the main engine firing, −0.03 with a side engine;
* +10 per leg in contact with the ground;
* **exactly +100 for coming to rest, exactly −100 for crashing or leaving the frame.**

That last line is the one the repository got wrong for a while. `landed` used to mean
`total_reward >= 200`, which is the threshold at which Gymnasium considers the *environment
solved on average* — a property of the task over many episodes, not a statement about any one
of them. The terminal reward answers the question directly, and that is what
`training/evaluate.py` reads. Both quantities are published, under two names, because on a
weaker policy they disagree.


### 4.1 Choosing the algorithm

`LunarLander-v3` is discrete, so DQN is eligible. PPO is used instead, for reasons that hold
on this environment:

1. **Stability.** The clipped objective bounds the size of a policy update, which matters on a
   dense reward where a single bad batch can undo a hundred thousand steps.
2. **Throughput.** PPO collects from vectorised environments natively. The configuration here
   uses 16 of them — in a `DummyVecEnv`, so they run in one process, sequentially, on one core.
   That still removes most of the per-episode overhead, and it is *not* a sixteen-fold speedup;
   the earlier version of this notebook claimed one. Use `use_subproc=True` in
   `make_train_env` for actual parallelism.
3. **It is the comparison that is made, not asserted.** Section 4.6 trains DQN at the same step
   budget and reports both. The previous version quoted reference scores from the SB3 RL Zoo —
   numbers produced on another machine, with another configuration, that nothing in this
   repository could check.

**Where the policy runs is a measurement, not a default.** Stable-Baselines3 warns that PPO
with an MLP policy belongs on the CPU. That warning was being suppressed by the test
configuration; with warnings turned into errors it surfaced, and the measurement confirmed it:
50 000 PPO steps in 14.9 s on the CPU against 20.3 s on an RTX 4060 Ti. DQN is the other way
round — 20 000 steps in 41.8 s on the GPU against 64.0 s — because it replays batches of 128
through two 256-unit layers, which is enough work to pay for the transfers. Both numbers are in
the comments on `PPOHyperParameters.device` and `DQNHyperParameters.device`.


### 4.2 Training, and what a run leaves behind

```powershell
uv run python -m rl_lander.training.train_lunarlander --algo ppo --seed 42
```

Each run gets its own directory, `models/<algo>/seed-<n>/`, holding:

| File | What it is |
| --- | --- |
| `best.zip` | the checkpoint `EvalCallback` kept — the best one seen during training |
| `final.zip` | the state training ended on |
| `training_curves.csv` | the rolling reward, sampled every 1000 steps |
| `manifest.json` | the metrics `best.zip` scored, on a fixed evaluation grid |

`best` and `final` are different policies, and the first version of this code conflated them:
`EvalCallback` wrote its checkpoint to `models/best_model.zip` — the same path for both
algorithms, so a DQN run overwrote a PPO one — and then `model.save()` wrote the *final* state
under a filename that said `best`. Every figure the repository published came from the final
state. In reinforcement learning the two differ, because performance oscillates late in
training.

Exactly one run becomes the shipped policy, through `scripts/publish_run.py`. It refuses a run
whose `best` is its `final` — a run no evaluation ever improved on.

How much that matters is not a matter of opinion here, because both checkpoints were kept and
both can be scored on the same grid. On the PPO run that is shipped, `final` beats `best` by
six points — `best` is best on the callback's own evaluation environment, not on every grid,
and on a run that does not collapse the two are within noise. On the DQN run of section 4.6,
`best` scores 271.13 and `final` scores **−610.34**: the policy at the end of training crashes
every episode. Under the old code that file would have been published, under a name saying
`best`, and every figure in the repository would have described it.


In [ ]:
from dataclasses import asdict

from rl_lander.training import PPOHyperParameters

pd.Series(asdict(PPOHyperParameters())).to_frame("value")


### 4.3 Five trainings, not one

The number this repository publishes is about the method. Five independent PPO runs, identical
hyper-parameters, seeds 42 to 46, each scored by the same protocol: 100 episodes on a fixed
evaluation grid, through the same `run_episodes` the exporter uses.

Two sources of variance, kept apart:

* **between trainings** — the spread of the five means below;
* **between episodes** — the spread inside one run, in `data/evaluation_summary.json`.

They answer different questions and adding them, or reporting one as the other, is how a method
looks more stable than it is.


In [ ]:
study = read_artifact("seed_study.json")
runs = pd.DataFrame(study["per_run"])

print(f"{study['n_runs']} runs of {study['total_timesteps']:,} steps, "
      f"evaluated on {study['evaluation']['n_episodes']} episodes from seed {study['evaluation']['seed']}")
print(f"mean of runs      : {study['mean_of_runs']}")
print(f"spread BETWEEN runs: {study['spread_between_runs']}")
print(f"worst / best run  : {study['worst_run']} / {study['best_run']}")
print(f"runs clearing 200 : {study['runs_clearing_200']} / {study['n_runs']}")
runs


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(runs["seed"].astype(str), runs["mean_reward"], color="#4c78a8", width=0.55)
ax.axhline(study["mean_of_runs"], color="#333333", linestyle="-", linewidth=1,
           label=f"mean of runs ({study['mean_of_runs']})")
ax.axhspan(study["mean_of_runs"] - study["spread_between_runs"],
           study["mean_of_runs"] + study["spread_between_runs"],
           color="#333333", alpha=0.12, label="+/- 1 sd between runs")
ax.axhline(200, color="#2ca02c", linestyle="--", label="solved threshold (200)")
ax.set_xlabel("Training seed")
ax.set_ylabel("Mean reward over 100 episodes")
ax.set_title("What changes when only the training seed changes")
ax.legend(loc="lower right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### 4.4 A learning curve is a band

`data/learning_curve_band.csv` holds the median of the five curves and their interquartile
range on a common timestep grid. A single trajectory says how one run went; it does not say
what training this configuration does.


In [ ]:
band = read_artifact("learning_curve_band.csv")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(band["timesteps"], band["median"], color="#4c78a8", label="median of runs")
ax.fill_between(band["timesteps"], band["q25"], band["q75"], color="#4c78a8", alpha=0.25,
                label="interquartile range")
ax.fill_between(band["timesteps"], band["min"], band["max"], color="#4c78a8", alpha=0.10,
                label="min-max")
ax.axhline(200, color="#2ca02c", linestyle="--", label="solved threshold (200)")
ax.set_xlabel("Timesteps")
ax.set_ylabel("Rolling mean reward during training")
ax.set_title(f"PPO on LunarLander-v3, {int(band['n_runs'].max())} runs")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


`ep_rew_mean` is a rolling mean over episodes finished *during* training, under a stochastic
policy that is still exploring. It is a progress signal and not the evaluation result — the
evaluation runs a deterministic policy on a fixed seed grid, and its number is section 5's.
The two are routinely confused, and the earlier version of this notebook plotted the first
under a title that suggested the second.

The same caution covers everything else TensorBoard shows (`uv run tensorboard --logdir
logs/tensorboard`). PPO's `policy_gradient_loss`, `value_loss` and `entropy_loss` are
optimisation diagnostics, not performance: the policy loss is a surrogate whose sign and
magnitude depend on the clipping and on the advantage normalisation, and it does **not**
decrease monotonically as the agent improves. A falling `value_loss` says the critic is
fitting the returns it currently sees, which is a statement about the critic and not about
the landing. `approx_kl` and `clip_fraction` are worth watching, because they say whether
the updates are staying inside the trust region -- but the only number that says how well
the agent lands is the evaluation of section 5.


### 4.5 One hyper-parameter at a time

The brief asks for one parameter changed at a time, so the effect can be isolated. Three of
them, at the same step budget, scored by the same protocol as the baseline.

The column that decides is the last one. Each trial is a **single** seed, and section 4.3
measured how much a run moves on the training seed alone. A difference smaller than that spread
is not evidence about the parameter — it is one draw from the same distribution.


In [ ]:
trials = read_artifact("hyperparameter_trials.csv")
trials


### 4.6 The DQN baseline, at an equal budget

The README used to announce `models/` as containing "best PPO / baseline DQN". There was only
the PPO. The baseline is trained here at the **same number of steps** as PPO: a comparison at
unequal budgets measures the budget.


In [ ]:
dqn = pd.DataFrame(study["dqn_at_equal_budget"])
if dqn.empty:
    print("No DQN run in models/. Train it: `uv run python scripts/train_study.py --only dqn-seed-42`.")
else:
    print(dqn.to_string(index=False))
    print(f"\nPPO at the same budget: mean of runs {study['mean_of_runs']}, "
          f"worst run {study['worst_run']}.")


## 5. Final evaluation

Everything below comes from one collection of episodes: the printed number, the CSV the
dashboard reads, the summary, and the manifest that says which policy and which code produced
them. The first version ran two evaluations with different reset semantics and published both —
261.4 at the root of the JSON, 262.2 under `metrics`, standard deviations 44 % apart — without
designating either as the result.


In [ ]:
summary = read_artifact("evaluation_summary.json")
metrics = summary["metrics"]

print(f"policy: {DEFAULT_MODEL_PATH.name}, evaluation grid seed {summary['canonical_seed']}")
print(f"mean reward   : {metrics['mean_reward']:.2f} +/- {metrics['std_reward']:.2f} "
      f"over {metrics['n_episodes']:.0f} episodes")
print(f"median / worst: {metrics['median_reward']:.2f} / {metrics['min_reward']:.2f}")
print(f"landing rate  : {metrics['landing_rate']:.0%}   (read from the terminal reward)")
print(f"above 200     : {metrics['threshold_rate']:.0%}   (a score, not a landing)")


In [ ]:
across = summary.get("across_seeds")
if across is None:
    print("Single evaluation grid. Re-run the exporter with several --seeds to see the spread.")
else:
    grids = pd.DataFrame(across["per_seed"])[
        ["seed", "mean_reward", "std_reward", "min_reward", "landing_rate"]
    ]
    print(f"{len(grids)} evaluation grids of the same policy:")
    print(grids.round(2).to_string(index=False))
    print(f"\nmean of means {across['mean_of_means']:.2f} +/- {across['spread_of_means']:.2f}")
    print(f"worst grid {across['worst_seed_mean']:.2f}, worst single episode {across['worst_episode']:.2f}, "
          f"lowest landing rate {across['lowest_landing_rate']:.0%}")


The table above is the reason the evaluation runs on more than one grid. The **mean** is stable
across grids; the dispersion, the worst episode and the landing rate are not. Publishing a
single grid's spread — and the repository published its most favourable one — makes the policy
look tighter than it is.


In [ ]:
episodes = pd.read_csv(EVALUATION_CSV)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].hist(episodes["total_reward"], bins=20, color="#4c78a8", edgecolor="white")
axes[0].axvline(200, color="#d62728", linestyle="--", label="200")
axes[0].set_title("Reward distribution")
axes[0].set_xlabel("Reward")
axes[0].set_ylabel("Episodes")
axes[0].legend()

counts = [
    int(((episodes["landed"] == 1) & (episodes["meets_threshold"] == 1)).sum()),
    int(((episodes["landed"] == 1) & (episodes["meets_threshold"] == 0)).sum()),
    int((episodes["landed"] == 0).sum()),
]
axes[1].bar(["landed,\nabove 200", "landed,\nbelow 200", "did not\nland"], counts,
            color=["#2ca02c", "#f0ad4e", "#d62728"])
axes[1].set_title("Landing and score are two questions")
axes[1].set_ylabel("Episodes")

total_firings = episodes["main_engine_firings"] + episodes["side_engine_firings"]
axes[2].scatter(total_firings, episodes["total_reward"], s=14, alpha=0.7, color="#4c78a8")
axes[2].set_title("Reward against engine use")
axes[2].set_xlabel("Total engine firings")
axes[2].set_ylabel("Reward")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


The middle panel is the distinction the code now keeps: a landing read from the environment's
terminal reward, and a score above the solved threshold. On this policy they nearly coincide,
which is exactly why conflating them was invisible — and why the check had to be a test rather
than a reading of the numbers.


## 6. What the repository ships

| Surface | Command | What it is for |
| --- | --- | --- |
| API | `uv run uvicorn rl_lander.api:app` | inference: `/play`, `/run`, `/reset`, `/info`, `/health`, `/ready` |
| Cockpit | `uv run streamlit run src/rl_lander/gui.py` | one episode, animated, with its metrics |
| Dashboard | `uv run streamlit run src/rl_lander/dashboard.py` | the evaluation run, behind filters |
| Video | `uv run python -m rl_lander.record_video` | `videos/landing.mp4` |

The reasoning behind that split — one service, thin frontends — is in `docs/architecture.md`.


## 7. Limits, and what I would do differently

**Five seeds is few.** It is enough to show that the seed matters and to publish a spread; it is
not enough to give that spread a confidence interval. Twenty would be the right number, and the
GPU time was not spent on it because the conclusion — publish the method's dispersion, not one
run's score — does not change past five.

**One seed per hyper-parameter trial.** The trials answer "is this table real", not "which value
is best". Reading a ranking out of them would repeat, one level up, the error the study exists to
correct. The `larger_than_seed_spread` column is there so the table cannot be misread.

**The evaluation grid is fixed at 100 episodes.** That is what the brief asks for. The
across-grid table in section 5 shows what the choice costs: the mean is stable, the tails are
not, and 100 episodes is not enough to say anything precise about the worst case — which, for a
landing autopilot, is the number that would actually matter.

**The task is nearly saturated.** The policy clears the threshold on every evaluation grid and
lands almost every episode. There is very little headroom left to distinguish configurations,
which is the real reason the hyper-parameter trials cannot separate anything: not that the
parameters do not matter, but that this environment, at this budget, is solved well enough that
their effect is smaller than the noise between seeds. A harder variant —
`enable_wind=True`, or a stochastic initial state — would make the comparison informative
again, and that is the experiment this repository is one step away from.
